In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.schema import Document
from langchain.retrievers import ParentDocumentRetriever
from langchain.memory import ConversationBufferMemory
from langchain.storage import InMemoryStore

load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

with open('sample.txt','r',encoding='utf-8') as f:
    raw_text=f.read()

parent_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

child_splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)

embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

vectorstore=FAISS.from_texts(["dummy"],embedding)
#vectorstore.delete([0])   # remove dummy

store=InMemoryStore()

parentdocretriever=ParentDocumentRetriever(vectorstore=vectorstore,docstore=store,child_splitter=child_splitter,parent_splitter=parent_splitter)

docs=[Document(page_content=raw_text)]
parentdocretriever.add_documents(docs)

memory=ConversationBufferMemory(memory_key='chat_history',return_messages=True)

C:\Users\DELL\anaconda3\envs\Langchain\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\DELL\anaconda3\envs\Langchain\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.ai.generativelanguage_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.ai.generativelanguage_v1beta past that date.
  warnings.warn(message, FutureWarning)
C:\Users\DELL\AppData\Local\Temp\ipykernel_6708\33259162

In [2]:
qa_chain=RetrievalQA.from_chain_type(llm=llm,retriever=parentdocretriever,memory=memory)

response1=qa_chain.invoke({'query':'Who created Langchain'})['result']
print('\nLLM answer: ',response1)


LLM answer:  LangChain was created by Harrison Chase.


In [3]:
qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.invoke,
    description='A basic LLM that answers users queries clearly'
)

agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_error=True
)

response2=agent.invoke('Who created Langchain')
print('\nLLM answer: ',response2)

C:\Users\DELL\AppData\Local\Temp\ipykernel_6708\4172796509.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agent(




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: LangChain was created by Harrison Chase.

> Finished chain.

LLM answer:  {'input': 'Who created Langchain', 'chat_history': [HumanMessage(content='Who created Langchain', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Who created Langchain', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={})], 'output': 'LangChain was created by Harrison Chase.'}


In [4]:
response3=parentdocretriever.get_relevant_documents("Who created LangChain?")
print('\nLLM answer: ',response3)

C:\Users\DELL\AppData\Local\Temp\ipykernel_6708\2146437805.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response3=parentdocretriever.get_relevant_documents("Who created LangChain?")



LLM answer:  [Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow')]
